Ran command: `python3 -c "
import json
with open('pytorch/chapter_deep-learning-computation/parameters.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i, cell in enumerate(nb['cells']):
    ctype = cell['cell_type']
    src = ''.join(cell['source'])
    print(f'=== Cell {i} ({ctype}) ===')
    print(src[:300] + ('...' if len(src) > 300 else ''))
"`
Ran command: `python3 -c "
import json
with open('pytorch/chapter_deep-learning-computation/parameters.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i in [0, 1, 2]:
    print(f'=== Cell {i} ===')
    print(''.join(nb['cells'][i]['source']))
"`

[parameters.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/parameters.ipynb) 是《动手学深度学习》中**深度学习计算**章节关于**参数管理（Parameter Management）**的核心教程。

该 Notebook 深入讲解了在模型构建完成后，如何**访问参数、初始化参数以及在层之间共享（绑定）参数**。以下是该章节的核心内容总结：

---

### 一、参数访问（Parameter Access）

深度学习中的参数属于 `torch.nn.parameter.Parameter` 类（继承自 `torch.Tensor`），它复合包含了**张量值（data）**、**梯度（grad）**以及系统跟踪所需的元信息。

#### 1. 访问特定层的参数
* **层状态字典**：`net[2].state_dict()` 返回该层包含的所有参数名及其对应张量（如权重 `weight` 和偏置 `bias`）。
* **提取参数对象与底层数据**：
  * 参数对象：`net[2].bias`（类型为 `Parameter`）。
  * 底层数值：`net[2].bias.data`（类型为 `Tensor`）。
  * 对应梯度：`net[2].weight.grad`（在未调用 `backward()` 前默认为 `None`）。

#### 2. 一次性访问/遍历所有参数
* **名称与参数迭代器**：
  * `net.named_parameters()`：递归产生元组 `(name, param)`，名称中包含分层索引（如 `'0.weight'`, `'2.bias'`）。
  * `net.parameters()`：仅产生参数张量迭代器。
* **全局状态字典访问**：
  * `net.state_dict()['2.bias'].data`。

#### 3. 嵌套块（Nested Blocks）的参数结构
* 当网络由多层 `nn.Sequential` 或自定义块嵌套时，PyTorch 会按树状层次结构组织参数命名（例如 `'block 1.0.weight'`）。
* 既支持像多维列表一样链式索引访问（例如 `rgnet[0][1][0].bias.data`），也支持统一用 `named_parameters()` 扁平展开查看。

---

### 二、参数初始化（Parameter Initialization）

框架默认会基于输入/输出维度按均匀分布初始化参数。为了保障数值稳定性与训练效率，需要灵活定制初始化策略。

#### 1. 内置初始化器（`torch.nn.init`）
通过定义初始化函数并配合 `net.apply(init_func)`，PyTorch 会递归遍历所有子模块：
* **高斯分布**：`nn.init.normal_(m.weight, mean=0, std=0.01)`，`nn.init.zeros_(m.bias)`
* **常数初始化**：`nn.init.constant_(m.weight, 1)`
* **Xavier 均匀初始化**：`nn.init.xavier_uniform_(m.weight)`

#### 2. 差异化与自定义初始化
* **局部差异化**：可以单独针对某些层调用不同的初始化函数，如 `net[0].apply(init_xavier)`、`net[2].apply(init_42)`。
* **自定义分布**：可以直接对 `m.weight.data` 进行就地张量运算。
  * 示例：抽样自 `[-10, 10]` 均匀分布并保留绝对值大于等于 5 的部分（即属于 `[-10, -5] ∪ [5, 10]`）：
    ```python
    def my_init(m):
        if isinstance(m, nn.Linear):
            nn.init.uniform_(m.weight, -10, 10)
            m.weight.data *= m.weight.data.abs() >= 5
    ```
* **手动硬编码**：直接就地修改底层张量切片：
  ```python
  net[0].weight.data[:] += 1
  net[0].weight.data[0, 0] = 42
  ```

---

### 三、参数绑定 / 共享（Parameter Binding）

在多层网络中，若希望两个或多个层共享同一套权重，只需**重复使用同一个层实例**：

```python
shared = nn.Linear(8, 8)
net = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(),
    shared,          nn.ReLU(),  # 第三层
    shared,          nn.ReLU(),  # 第五层
    nn.Linear(8, 1)
)
```

#### 关键特性与机制：
1. **内存共用**：`net[2].weight` 与 `net[4].weight` 指向同一内存地址，任何一处的修改都会同步作用于另一处。
2. **反向传播梯度累加**：
   在反向传播计算梯度时，这两个层位置上产生的损失梯度会**自动相加**（`grad = grad_layer2 + grad_layer4`），从而确保共享权重的更新正确兼顾所有使用位置。

---

### 四、核心 API 总结表

| 操作需求 | 常用方法 / 属性 | 说明 |
| :--- | :--- | :--- |
| **按层访问参数** | `layer.weight` / `layer.bias` | 得到 `Parameter` 实例 |
| **访问数值张量** | `param.data` | 获取底层 `Tensor`，用于读取或就地更新 |
| **查看参数梯度** | `param.grad` | 获取反向传播计算出的梯度张量 |
| **递归遍历所有参数** | `net.named_parameters()` | 遍历所有 `(name, param)` 元组 |
| **递归应用初始化** | `net.apply(fn)` | 遍历各子模块 `m` 执行自定义配置函数 |
| **权重共享** | 复用同个 `nn.Module` 实例 | 反向传播时多处梯度会自动累加 |

In [1]:
import torch
from torch import nn

In [2]:
net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))
X = torch.rand(size=(2,4))

net(X)

tensor([[0.2983],
        [0.1472]], grad_fn=<AddmmBackward0>)

In [3]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[ 0.2607, -0.0895, -0.0018,  0.2862,  0.0707,  0.3317,  0.1538,  0.1393]])), ('bias', tensor([0.0049]))])


In [4]:
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>


In [5]:
print(net[2].bias)

Parameter containing:
tensor([0.0049], requires_grad=True)


In [6]:
print(net[2].bias.data)

tensor([0.0049])


In [7]:
net[2].weight.grad == None

True

In [8]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))


In [9]:
print(*[(name, param.shape) for name, param in net.named_parameters()])

('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [13]:
net.state_dict()['2.bias'].data

tensor([0.0049])

In [14]:
print(net.state_dict()['2.bias'])

tensor([0.0049])


In [15]:
def block1():
    return nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1())
    return net

In [16]:
rgnet = nn.Sequential(block2(), nn.Linear(4,2))
rgnet(X)

tensor([[0.1533, 0.0485],
        [0.1533, 0.0485]], grad_fn=<AddmmBackward0>)

In [17]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=2, bias=True)
)


In [18]:
rgnet[0][1][0].bias.data

tensor([ 0.4944, -0.1123, -0.0350,  0.0705,  0.2751,  0.2525, -0.0484,  0.4673])

In [20]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std = 0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0002, -0.0066, -0.0009,  0.0001]), tensor(0.))

In [22]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [24]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)

print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.4025,  0.4755, -0.2610, -0.1833])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [26]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]



Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-5.3248,  0.0000,  0.0000, -8.6945],
        [ 9.4657, -0.0000, -9.6535, -0.0000]], grad_fn=<SliceBackward0>)

In [27]:
net[0].weight.data[:] = 1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.,  1.,  1.,  1.])

In [28]:
shard = nn.Linear(8,8)
net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), 
        shard, nn.ReLU(), 
        shard, nn.ReLU(),
        nn.Linear(8, 4))

print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])



tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


Searched for "FancyMLP"
Searched for "FancyMLP"
Listed directory chapter_deep-learning-computation
Ran command: `python3 -c "
import json
with open('pytorch/chapter_deep-learning-computation/model-construction.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i, cell in enumerate(nb['cells']):
    src = ''.join(cell['source'])
    if 'class ' in src:
        print(f'Cell {i}:')
        print(src)
"`

针对 [parameters.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/parameters.ipynb) 末尾的 4 道练习题，以下是详细的解答与代码实现：

---

### 练习 1：使用 `FancyMLP` 模型访问各个层的参数

> **注**：在原书 MXNet/TF 版本中该模型名为 `FancyMLP`，而在 PyTorch 版的 [model-construction.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_deep-learning-computation/model-construction.ipynb) 中对应实现为 `FixedHiddenMLP`（包含常量固定权重、可学习线性层以及控制流）。

#### 1. 模型定义回顾
```python
import torch
from torch import nn
from torch.nn import functional as F

class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机常量张量（非 Parameter）
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用线性层（参数共享）
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
```

#### 2. 参数访问方式
```python
net = FixedHiddenMLP()

# 方法 1：通过属性直接访问可学习层的参数和常量张量
print("Linear 权重形状:", net.linear.weight.shape)
print("Linear 偏置形状:", net.linear.bias.shape)
print("常量 rand_weight 形状:", net.rand_weight.shape)

# 方法 2：使用 named_parameters() 遍历可训练参数
print("\n--- 可学习参数列表 ---")
for name, param in net.named_parameters():
    print(f"参数名: {name:15s} | 形状: {str(list(param.shape)):10s} | requires_grad: {param.requires_grad}")

# 说明：
# net.rand_weight 只是普通的 torch.Tensor，不是 nn.Parameter，因此不会被 named_parameters() 捕获。
# 如果希望框架将其作为模型状态保存但不需要梯度，可以使用 self.register_buffer('rand_weight', ...) 注册，
# 注册后可通过 net.named_buffers() 统一访问。
```

---

### 练习 2：查看初始化模块文档以了解不同的初始化方法

PyTorch 的核心初始化工具集中在 `torch.nn.init` 模块中，常用初始化方法及适用场景整理如下：

| 初始化方法 | API 调用 | 适用场景 / 理论背景 |
| :--- | :--- | :--- |
| **高斯分布** | `nn.init.normal_(w, mean=0.0, std=0.01)` | 通用浅层网络，通过较小的方差防止初始激活值饱和 |
| **均匀分布** | `nn.init.uniform_(w, a=-0.1, b=0.1)` | 限制初始参数在对称区间 `[a, b]` 内 |
| **常数初始化** | `nn.init.constant_(w, val)` / `zeros_` / `ones_` | 常用在偏置向量 `bias` 初始化为 0，或残差模块的分支缩放系数 |
| **Xavier (Glorot) 初始化** | `nn.init.xavier_uniform_(w)`<br>`nn.init.xavier_normal_(w)` | 适合 **Tanh、Sigmoid** 等对称激活函数；根据输入输出维度保持各层方差稳定 |
| **Kaiming (He) 初始化** | `nn.init.kaiming_uniform_(w, nonlinearity='relu')`<br>`nn.init.kaiming_normal_(w, nonlinearity='relu')` | 专门针对 **ReLU、LeakyReLU** 等不对称且将负数置 0 的非线性激活函数 |
| **正交初始化 (Orthogonal)** | `nn.init.orthogonal_(w, gain=1.0)` | 保证权重矩阵行/列正交，保持向量模长，常用于 **RNN / LSTM** 防止梯度消失或爆炸 |
| **截断正态 (Truncated Normal)** | `nn.init.trunc_normal_(w, mean=0, std=0.02, a=-2, b=2)` | 舍弃超过特定方差范围的尾部极端值，广泛应用于 **Transformer、ViT** 的全连接与 Patch 投影层 |

---

### 练习 3：构建包含共享参数层的多层感知机并观察参数与梯度

#### 1. 实验代码
```python
import torch
from torch import nn

# 1. 构造共享层
shared_layer = nn.Linear(8, 8)

# 2. 构建 MLP：在第 2 层和第 4 层共享同一个 Linear 实例
net = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(),
    shared_layer,    nn.ReLU(),  # net[2]
    shared_layer,    nn.ReLU(),  # net[4] (共享层)
    nn.Linear(8, 1)
)

# 3. 准备虚拟输入与标签
X = torch.randn(size=(4, 4))
y = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

# 4. 前向传播与损失计算
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

optimizer.zero_grad()
out = net(X)
loss = criterion(out, y)
loss.backward()

# 5. 观察共享层参数与其梯度
print("--- 检查参数内存是否共享 ---")
print("net[2] 与 net[4] 的 weight 是否相同对象:", net[2].weight is net[4].weight)
print("net[2] 与 net[4] 的 bias 是否相同对象:  ", net[2].bias is net[4].bias)

print("\n--- 检查梯度 ---")
print("共享层 weight 的梯度形状:", net[2].weight.grad.shape)
print("共享层 bias 的梯度值:\n", net[2].bias.grad)
```

#### 2. 梯度行为观察与解析
* **参数地址相同**：`net[2].weight is net[4].weight` 输出为 `True`，它们指向同一块物理显存/内存地址。
* **梯度自动累加**：
  在反向传播链式法则中，设总损失为 `L`，该共享层在第 2 层处的输出贡献了梯度 `∂L / ∂W_(pos1)`，在第 4 层处贡献了梯度 `∂L / ∂W_(pos2)`。最终共享权重的总梯度满足：
  ```text
  W_shared.grad = (∂L / ∂W_(pos1)) + (∂L / ∂W_(pos2))
  ```
  PyTorch 在反向遍历计算图遇到相同张量节点时，会自动将回传的梯度累加到 `param.grad` 中。

---

### 练习 4：为什么共享参数是个好主意？

在深度学习模型设计中，参数共享是一项关键的**归纳偏置（Inductive Bias）**与优化手段，其主要优势包括：

1. **利用数据结构的对称性与先验特征**
   * **平移不变性（CNN）**：卷积核在整幅图像的不同空间位置滑动并共享同一套权重，利用了局部视觉模式在空间平移下的不变性。
   * **时间不变性与序列延续（RNN / GRU / LSTM）**：每个时间步共用同一组状态转移矩阵，使模型能够处理任意长度的序列输入，并捕捉统一的时间序列动力学规律。
2. **显著降低参数量与内存消耗**
   * 极大地降低了模型尺寸和存储占用，不仅降低了部署成本，也允许构建层数更深的网络架构（如 **ALBERT** 跨层共享 Transformer 的注意力层与前馈网络，参数量仅为 BERT 的一小部分，却能达到接近的表现）。
3. **强大的正则化效应，防止过拟合**
   * 自由度（自由参数量）大幅降低，强迫共享模块学到更具通用性、跨上下文的稳健表征，从而在小样本场景下提升泛化性能。
4. **统一输入与输出的特征空间（Weight Tying）**
   * 在现代语言模型（如 GPT、BERT、Word2Vec）中，输入层的 Word Embedding 矩阵与最后一层将隐层投影回词表预测的 Linear 权重矩阵往往是转置共享的。这不仅节省了千万量级的词表参数，还能强制保证词嵌入与输出分类投影处于同一个几何语义空间中。